# Homework 2: Multiple Linear Regression and Logistic Regression


##  Part 1: Linear Model (Publication Dataset)

### Filter the dataset to only clinical trials that have been published

In [ ]:
import numpy as np
from ISLP import load_data

import statsmodels.api as sm

from statsmodels.stats.outliers_influence \
import variance_inflation_factor as VIF
from statsmodels.stats.anova import anova_lm

from ISLP.models import (ModelSpec as MS,
summarize,
poly)

def abline(ax, b, m, *args, **kwargs):
    "Add a line with slope m and intercept b to ax"
    xlim = ax.get_xlim()
    ylim = [m * xlim[0] + b, m * xlim[1] + b]
    ax.plot(xlim, ylim, *args, **kwargs)

publication = load_data('Publication')
newPublication = publication[publication['status'] == 1]  # Filter the dataset to only clinical trials that have been published
newPublication['mech'] = newPublication['mech'].cat.remove_unused_categories() # Remove used category

                            OLS Regression Results                            
Dep. Variable:                 impact   R-squared:                       0.507
Model:                            OLS   Adj. R-squared:                  0.454
Method:                 Least Squares   F-statistic:                     9.588
Date:                Sun, 13 Apr 2025   Prob (F-statistic):           3.20e-15
Time:                        20:59:23   Log-Likelihood:                -611.04
No. Observations:                 156   AIC:                             1254.
Df Residuals:                     140   BIC:                             1303.
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
intercept         33.2210      3.852      8.

/var/folders/xm/1b87kczn06db1trrs93nstzm0000gn/T/ipykernel_95445/1414965759.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  newPublication['mech'] = newPublication['mech'].cat.remove_unused_categories() # Remove used category


### Fit a linear regression that models impact as a function of all other variables.

In [9]:

x1 = MS(['posres', 'multi', 'clinend', 'mech', 'sampsize', 'budget','time']).fit_transform(newPublication)
y = newPublication['impact']

model1 = sm.OLS(y, x1)
results1 = model1.fit()

print(results1.summary())

                            OLS Regression Results                            
Dep. Variable:                 impact   R-squared:                       0.578
Model:                            OLS   Adj. R-squared:                  0.526
Method:                 Least Squares   F-statistic:                     11.11
Date:                Sun, 13 Apr 2025   Prob (F-statistic):           1.77e-18
Time:                        20:59:28   Log-Likelihood:                -598.91
No. Observations:                 156   AIC:                             1234.
Df Residuals:                     138   BIC:                             1289.
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
intercept         21.3248      4.391      4.

### Based on the summary of this model should you keep all of the predictors in the model? If not, which should you drop and why? 

No because there are some predictors that are not statistically significant by looking at their p value. Therefore, we should drop
predictors where the p-value is greater than 0.05. These include 'multi', 'mech', 'sampsize', 'posres' and 'budget'

### fit a new model that does not include them and assess whether it is better than the original. 

In [11]:
x2 = MS(['clinend', 'time']).fit_transform(newPublication)

model2 = sm.OLS(y,x2)
results2 = model2.fit()

print(results2.summary())




                            OLS Regression Results                            
Dep. Variable:                 impact   R-squared:                       0.475
Model:                            OLS   Adj. R-squared:                  0.468
Method:                 Least Squares   F-statistic:                     69.16
Date:                Sun, 13 Apr 2025   Prob (F-statistic):           4.02e-22
Time:                        21:03:13   Log-Likelihood:                -615.93
No. Observations:                 156   AIC:                             1238.
Df Residuals:                     153   BIC:                             1247.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
intercept     20.1241      2.141      9.401      0.0

The new model isn't any better than the first brcause the first model has an adjusted R-squared of 0.526, and the second model has a lower adjusted R-squared of 0.468.

### Next, take a look at the correlation between your variables. Do you see any issues? Which variables are most highly correlated? Investigate and describe the relationship between the two sets of highest correlated variables. 

In [12]:
corr = MS(['posres', 'multi', 'clinend', 'sampsize', 'budget', 'time']).fit_transform(newPublication) 
corr_matrix = corr.corr()
print(corr_matrix)

           intercept    posres     multi   clinend  sampsize    budget  \
intercept        NaN       NaN       NaN       NaN       NaN       NaN   
posres           NaN  1.000000 -0.242713 -0.347171 -0.232702 -0.177944   
multi            NaN -0.242713  1.000000  0.445261  0.181366  0.369388   
clinend          NaN -0.347171  0.445261  1.000000  0.507468  0.345001   
sampsize         NaN -0.232702  0.181366  0.507468  1.000000  0.594168   
budget           NaN -0.177944  0.369388  0.345001  0.594168  1.000000   
time             NaN  0.124236 -0.298522 -0.400694 -0.231833 -0.198679   

               time  
intercept       NaN  
posres     0.124236  
multi     -0.298522  
clinend   -0.400694  
sampsize  -0.231833  
budget    -0.198679  
time       1.000000  


Some predictor pairs exhibit high correlations, potentially causing multicollinearity that obscures the distinct impact of each variable. In particular, the 'sampsize' and 'budget' variables show the strongest correlation (0.594168). This relationship is intuitive since experiments with larger sample sizes generally require higher budgets due to the increased costs associated with managing more participants. The second highest correlation is observed between 'clinend' and 'sampsize' (0.507468). This suggests that trials focusing on a clinical endpoint tend to involve larger sample sizes. This makes sense because studies designed to evaluate specific clinical outcomes usually require a broader range of participants to capture the necessary diversity.

### Fit another linear regression model that includes all variables except for one in each pair of most highly correlated variables. How well does this model fit the data? How does this compare to your earlier models? 

In [ ]:
x3 = MS(['posres', 'multi', 'mech', 'budget','time']).fit_transform(newPublication)
model3 = sm.OLS(y,x3)
results3 = model3.fit()
print(results3.summary())


Fitting a new regression model by removing one variable from each pair of highly correlated predictors yields mixed results:
Excluding just 'sampsize':
The adjusted R-squared increased slightly from 0.526 to 0.529, indicating a modest improvement in model fit.
Excluding both 'budget' and 'sampsize':
The adjusted R-squared remained essentially unchanged at 0.526, suggesting no improvement over the original model.
Excluding 'budget' and 'clinend':
The adjusted R-squared dropped significantly to 0.458, meaning the model’s explanatory power worsened.
Excluding 'clinend' and 'sampsize':
The adjusted R-squared further decreased to 0.454, indicating an even poorer fit.

In summary, while removing 'sampsize' alone provides a slight improvement over the original model, eliminating other combinations of predictors results in a noticeably poorer fit compared to earlier models.
model worse slightly: the adjusted R-squa


## Part 2: Logistic Regression (Phil's Shadow Predictions)

### Start by filtering the data to only observations where Phil saw a "Full Shadow" or "No Shadow", and observations without NA values. 

In [2]:
import pandas as pd

phil = pd.read_csv('phil.csv')
phil_clean = phil[phil['Punxsutawney Phil'].isin(['Full Shadow', 'No Shadow'])].dropna().copy()
phil_clean['Punxsutawney Phil'] = (phil_clean['Punxsutawney Phil'] == 'No Shadow').astype(int)
y_logit = phil_clean['Punxsutawney Phil']

### Now, model Phil's predictions for spring ("Full Shadow" -> two more weeks of winter, "No Shadow" -> early spring). Make one model for overall average, one for the Northeast, one for the Midwest, and one for Pennsylvania.

In [ ]:
# Overall
x_overall = sm.add_constant(phil_clean[['February Average Temperature', 'March Average Temperature']])
model_overall = sm.Logit(y_logit, x_overall).fit()
model_overall.summary()

In [ ]:
# Northeast
x_ne = sm.add_constant(phil_clean[[ 
    'February Average Temperature (Northeast)', 
    'March Average Temperature (Northeast)']])
model_ne = sm.Logit(y_logit, x_ne).fit()
model_ne.summary()

In [ ]:
# Midwest
x_mw = sm.add_constant(phil_clean[[ 
    'February Average Temperature (Midwest)', 
    'March Average Temperature (Midwest)']])
model_mw = sm.Logit(y_logit, x_mw).fit()
model_mw.summary()

In [ ]:
# Pennsylvania
x_pa = sm.add_constant(phil_clean[[ 
    'February Average Temperature (Pennsylvania)', 
    'March Average Temperature (Pennsylvania)']])
model_pa = sm.Logit(y_logit, x_pa).fit()
model_pa.summary()

### What do your models tell you about Phil's predictions across various regions? Do you think Phil is good at his job of predicting spring? 


The Northeast, Midwest, and Pennsylvania models all exhibit very low pseudo R-squared values (0.006570, 0.008016, and 0.003810, respectively), indicating that these models explain only a minimal amount of the variability in the outcomes. Moreover, their high LLR p-values (0.7464, 0.6998, and 0.8440) suggest that these models are not significantly better than a null model with no predictors.

In contrast, the overall model has an LLR p-value of 0.04235, meaning that it is statistically significantly better than a null model and that the observed relationship is unlikely due to chance. This improvement may stem from the overall temperature measures effectively “smoothing out” the noise present in the regional data by capturing a broader trend.

However, even the overall model has a low pseudo R-squared value of 0.07100, similar to the regional models, which indicates that it still explains only a limited portion of the variation in the response variable. This limited explanatory power is further underscored by the imbalanced dataset, where the majority of observations correspond to 'Full Shadow', potentially biasing the model toward predicting that outcome.